## Validate running predictions using ICEBERG model

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import torch
import re
from tqdm import tqdm

In [2]:
# --- 1. SETUP MS-PRED ---
# Ensure this points to your ms-pred folder
sys.path.append('/data/nas-gpu/wang/tmach007/ms-pred')

In [3]:
try:
    import ms_pred.common as common
    from ms_pred.dag_pred.iceberg_elucidation import (
        iceberg_prediction, 
        load_global_config, 
        load_pred_spec
    )
except ImportError:
    print("CRITICAL ERROR: Could not import 'ms_pred'. Check path.")
    sys.exit(1)

In [4]:
# MATCHMS
from matchms import Spectrum
from matchms.similarity import ModifiedCosine

In [5]:
# --- 2. CONFIGURATION ---
CONFIG = {
    # Input: The novel MoNA pairs generated in the previous step
    'input_tsv': '/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/eda/novel_mona_msg_pairs.tsv',
    
    # Metadata Source: MassSpecGym data (Updated from CASMI)
    'spec_data_path': '/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df_spec_sim.pkl',
    
    # ICEBERG config
    'iceberg_yaml': '/data/nas-gpu/wang/tmach007/ms-pred/configs/iceberg/iceberg_elucidation.yaml', # Update if needed
    
    'output_dir': '/data/nas-gpu/wang/tmach007/ms-pred/notebooks/run_predictions/results/iceberg_validation_exact/',
    'tolerance': 0.05,
    'gpu_id': 0
}

In [16]:
# --- 3. HELPER FUNCTIONS ---
def clean_adduct(adduct_str):
    """ Standardize adduct string for ICEBERG """
    if pd.isna(adduct_str): return '[M+H]+'
    if adduct_str == '[M+NA]+': return '[M+Na]+'
    return str(adduct_str).strip()

def parse_nce(nce_val):
    """ Extract numeric collision energy from string or float """
    if pd.isna(nce_val): return 30.0 # Standard fallback
    if isinstance(nce_val, (int, float)): return float(nce_val)
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", str(nce_val))
    if nums: return float(nums[0])
    return 30.0

def to_matchms_spectrum(peaks, prec_mz):
    """ Convert peak list to matchms Spectrum object """
    if not peaks or not isinstance(peaks, list): return None
    mzs, ints = zip(*peaks)
    
    # matchms requires mz to be sorted
    mz_arr = np.array(mzs, dtype=float)
    int_arr = np.array(ints, dtype=float)
    sort_idx = np.argsort(mz_arr)

    return Spectrum(mz=mz_arr[sort_idx], 
                    intensities=int_arr[sort_idx], 
                    metadata={'precursor_mz': float(prec_mz)})

In [17]:
# --- 4. MAIN EXECUTION ---
def run_iceberg_prediction_only():
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    device = f"cuda:{CONFIG['gpu_id']}" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    local_config = load_global_config(CONFIG['iceberg_yaml'])
    local_config['device'] = device
    local_config['nce'] = True

    print(f"Loading queries from {CONFIG['input_tsv']}...")
    queries_df = pd.read_csv(CONFIG['input_tsv'], sep='\t')
    
    print(f"Loading metadata from {CONFIG['spec_data_path']}...")
    spec_df = pd.read_pickle(CONFIG['spec_data_path'])
    spec_lookup = spec_df.set_index('spec_id')

    results = []
    unique_queries = queries_df.drop_duplicates(subset=['mona_smiles'])
    
    print(f"Starting ICEBERG predictions for {len(unique_queries)} molecules...")
    
    for idx, row in tqdm(unique_queries.iterrows(), total=len(unique_queries)):
        mona_id = row['mona_id']
        smiles = row['mona_smiles']
        
        if mona_id not in spec_lookup.index: continue
            
        meta_row = spec_lookup.loc[mona_id]
        raw_nce = meta_row.get('nce_updated', meta_row.get('nce'))
        nce = parse_nce(raw_nce)
        adduct = clean_adduct(meta_row.get('prec_type'))
        
        try:
            original_cwd = os.getcwd() 
            
            try:
                os.chdir('/data/nas-gpu/wang/tmach007/ms-pred')
                
                # 1. Run ICEBERG Prediction
                res_path, pmz = iceberg_prediction(
                    candidate_smiles=[smiles], 
                    collision_energies=[nce], 
                    adduct=adduct, 
                    **local_config
                )
                
                # 2. Load the predicted spectra using the author's function
                pred_smiles, pred_specs, pred_frags = load_pred_spec(
                    load_dir=res_path, 
                    merge_spec=local_config.get('merge_spec', False), 
                    denoise_spectrum=local_config.get('denoise_spectrum', False)
                )
                
                # 3. Extract the peaks for our specific collision energy
                predicted_peaks = []
                if len(pred_specs) > 0:
                    # pred_specs is a list of dictionaries. We take the first (and only) molecule.
                    spec_dict = pred_specs[0]
                    
                    # ICEBERG stores it under a stringified key, or 'nan' if merged
                    target_key = 'nan' if local_config.get('merge_spec', False) else str(float(nce))
                    
                    if target_key in spec_dict:
                        # Convert 2D array [mz, inten] back to list of tuples
                        raw_peaks = spec_dict[target_key]
                        predicted_peaks = [(float(p[0]), float(p[1])) for p in raw_peaks if p[1] > 0.005]
                    else:
                        # Fallback: just grab the first available spectrum if exact key fails
                        first_key = list(spec_dict.keys())[0]
                        raw_peaks = spec_dict[first_key]
                        predicted_peaks = [(float(p[0]), float(p[1])) for p in raw_peaks if p[1] > 0.005]

            finally:
                os.chdir(original_cwd) 
                
        except Exception as e:
            print(f"Prediction/Extraction failed for {mona_id}: {e}")
            predicted_peaks = []
            pmz = [meta_row.get('prec_mz')]

        results.append({
            'mona_id': mona_id,
            'smiles': smiles,
            'applied_nce': nce,
            'applied_adduct': adduct,
            'precursor_mz': pmz[0] if isinstance(pmz, (list, np.ndarray)) else pmz,
            'predicted_peaks': predicted_peaks
        })

    out_pkl = os.path.join(CONFIG['output_dir'], 'iceberg_raw_predictions.pkl')
    pd.DataFrame(results).to_pickle(out_pkl)
    print(f"\nSuccessfully generated {len(results)} spectra.")
    print(f"Predictions saved to: {out_pkl}")

In [18]:
run_iceberg_prediction_only()

Using device: cuda:0
Loading queries from /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/eda/novel_mona_msg_pairs.tsv...
Loading metadata from /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df_spec_sim.pkl...
Starting ICEBERG predictions for 13 molecules...


  0%|          | 0/13 [00:00<?, ?it/s]

CUDA_VISIBLE_DEVICES=0,1 /data/nas-gpu/wang/tmach007/ms-pred/y/envs/ms-gen/bin/python src/ms_pred/dag_pred/predict_smis.py \
               --batch-size 32 \
               --num-workers 8 \
               --dataset-labels /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/e08a736a8796e576d72af6e712f9a699/cands_df_iceberg_elucidation.tsv \
               --sparse-out \
               --sparse-k 100 \
               --max-nodes 100 \
               --threshold 0.0 \
               --gen-checkpoint /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_generate.ckpt \
               --inten-checkpoint /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_score.ckpt \
               --save-dir /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/e08a736a8796e576d72af6e712f9a699 \
               --adduct-shift \
               --gpu


/data/nas-gpu/wang/tmach007/ms-pred/y/envs/ms-gen/lib/python3.8/site-packages/pytorch_lightning/utilities/seed.py:48: LightningDeprecationWarning: `pytorch_lightning.utilities.seed.seed_everything` has been deprecated in v1.8.0 and will be removed in v2.0.0. Please use `lightning_fabric.utilities.seed.seed_everything` instead.
  rank_zero_deprecation(
Global seed set to 42
Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file weights/canopus_iceberg_score.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file weights/canopus_iceberg_generate.ckpt`


2026-02-19 20:14:12,354 INFO: 
adduct_shift: true
batch_size: 32
binned_out: false
dataset_labels: /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/e08a736a8796e576d72af6e712f9a699/cands_df_iceberg_elucidation.tsv
dataset_name: null
debug: false
gen_checkpoint: /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_generate.ckpt
gpu: true
inten_checkpoint: /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_score.ckpt
max_nodes: 100
num_bins: 15000
num_workers: 8
save_dir: /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/e08a736a8796e576d72af6e712f9a699
seed: 42
sparse_k: 100
sparse_out: true
split_name: split_22.tsv
subset_datasets: none
threshold: 0.0
upper_limit: 1500

2026-02-19 20:14:12,476 INFO: Loaded gen / inten models from /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_generate.ckpt & /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_score.ckpt
2026-02-19 20:14:12,477 INFO: Preparing entries


...

2026-02-19 20:14:24,754 INFO: 
adduct_shift: true
batch_size: 32
binned_out: false
dataset_labels: /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/cf2869e3acdb10feabac940451189b0b/cands_df_iceberg_elucidation.tsv
dataset_name: null
debug: false
gen_checkpoint: /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_generate.ckpt
gpu: true
inten_checkpoint: /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_score.ckpt
max_nodes: 100
num_bins: 15000
num_workers: 8
save_dir: /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/cf2869e3acdb10feabac940451189b0b
seed: 42
sparse_k: 100
sparse_out: true
split_name: split_22.tsv
subset_datasets: none
threshold: 0.0
upper_limit: 1500

2026-02-19 20:14:24,874 INFO: Loaded gen / inten models from /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_generate.ckpt & /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_score.ckpt
2026-02-19 20:14:24,875 INFO: Preparing entries


  0%|          | 0/1 [00:00<?, ?it/s]

2026-02-19 20:14:30,441 INFO: There are 1 entries to process


100%|██████████| 13/13 [00:25<00:00,  1.93s/it]



Successfully generated 13 spectra.
Predictions saved to: /data/nas-gpu/wang/tmach007/ms-pred/notebooks/run_predictions/results/iceberg_validation_exact/iceberg_raw_predictions.pkl


In [21]:
pred = pd.read_pickle("/data/nas-gpu/wang/tmach007/ms-pred/notebooks/run_predictions/results/iceberg_validation_exact/iceberg_raw_predictions.pkl")

In [22]:
len(pred)

13

In [32]:
pred.tail()

,mona_id,smiles,applied_nce,applied_adduct,precursor_mz,predicted_peaks
8,MoNA_38543,C=C1C(=O)OC2CC(C)=C(C(C)CCCOC3OC(CO)C(O)C(O)C3...,62.847015,[M+NA]+,451.194000,[]
9,MoNA_42473,COc1cc(C)c2c(=O)c3c(O)cc(OC)cc3oc2c1,62.847015,[M+H]+,287.091400,[]
10,MoNA_77024,CC1=CC2(O)OC3(CC2=C(C)C)C(C)CCC13,138.198340,[M+H]+,235.169256,[]
11,MoNA_2766,CC1=C2C3OC(=O)C(C)C3(O)CCC2(C)CCC1=O,62.847015,[M+H]+,265.143436,[]
12,MoNA_42489,Cc1cc(O)cc2oc3c(Cl)c(O)cc(O)c3c(=O)c12,62.847015,[M+H]+,293.021128,[]
